In [46]:
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"


tokenizer = NllbTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)






Loading weights: 100%|██████████| 512/512 [00:00<00:00, 542.28it/s, Materializing param=model.shared.weight]                                   
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [47]:
from ultralytics import YOLO
from manga_ocr import MangaOcr
import cv2
from PIL import Image
from transformers import pipeline
import torch

In [48]:
def translate_ja_to_ar(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        src_lang="jpn_Jpan"
    )

    arb_id = tokenizer.convert_tokens_to_ids("eng_Latn")

    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=arb_id,
        max_length=256
    )

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

print(translate_ja_to_ar("私は学生です。"))

I am a student.


In [49]:
yolomodel = YOLO("models/comic-speech-bubble-detector.pt")
ocr = MangaOcr()

2026-02-19 21:38:48.330 | INFO     | manga_ocr.ocr:__init__:16 - Loading OCR model from kha-white/manga-ocr-base
Loading weights: 100%|██████████| 264/264 [00:00<00:00, 515.24it/s, Materializing param=encoder.pooler.dense.weight]                                       
The tied weights mapping and config for this model specifies to tie decoder.bert.embeddings.word_embeddings.weight to decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie decoder.cls.predictions.bias to decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
MangaOcrModel LOAD REPORT from: kha-white/manga-ocr-base
Key                                  | Status     |  | 
------------------

In [50]:
image_file = "img/006.jpg"

In [51]:
img = cv2.imread(image_file)

In [52]:
results = yolomodel(img, conf=0.3)


0: 1024x736 6 text_bubbles, 1812.0ms
Speed: 33.9ms preprocess, 1812.0ms inference, 20.7ms postprocess per image at shape (1, 3, 1024, 736)


In [53]:
print("\nReading text from bubbles:\n")
bubble_number = 1


Reading text from bubbles:



In [54]:
for box in results[0].boxes:
    # Get coordinates
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    
    # Crop the bubble
    bubble_crop = img[y1:y2, x1:x2]
    output_path = f"crop{bubble_number-1}.jpg"

    
    #change to pil format
    bubble_pil = Image.fromarray(bubble_crop)
    
    # Read Japanese text
    text = ocr(bubble_pil)
    print(f"Bubble {bubble_number}:")
    print(text)
    print(translate_ja_to_ar(text))
    print("─" * 40)
    bubble_number += 1

print(f"\nFinished! Found and read {bubble_number-1} bubbles.")

Bubble 1:
あっもしかしてお手伝いの方ですか？
Can you help me?
────────────────────────────────────────
Bubble 2:
ようこそ！
Welcome to the house!
────────────────────────────────────────
Bubble 3:
風船いっばい夢いっぱいの天文部へ！
The ship is full of dreams!
────────────────────────────────────────
Bubble 4:
すっすみません今散らかってて
Sorry, I'm not here.
────────────────────────────────────────
Bubble 5:
な、なんだこれ！？
What is this?
────────────────────────────────────────
Bubble 6:
．．．風船？
...about the ship?
────────────────────────────────────────

Finished! Found and read 6 bubbles.


In [55]:
annotated_image = results[0].plot()

In [56]:
output_path = "detected.jpg"
cv2.imwrite(output_path, annotated_image)
print(f"Saved result to: {output_path}")

Saved result to: detected.jpg
